# Distribution Planning Benchmark

Goal: test whether the policy transports a population distribution toward a target law while paying only the movement cost needed to do so.

**Environment Basics**

- State space: $\mathcal{X}=\{0,\ldots,n_x-1\}$, arranged on a ring.
- Action space: three actions encoded as `{0, 1, 2}` and mapped to moves $\{-1,0,+1\}$ modulo $n_x$.
- Population law: $\mu_t$ is a probability vector over ring sites; $\mu_\star$ is the target distribution.
- Dynamics: choosing move $a\in\{-1,0,+1\}$ sends state $x$ to $(x+a)\bmod n_x$.
- Main task: match $\mu_\star$ while limiting unnecessary movement around the ring.

The ring dynamics use actions $a\in\{-1,0,+1\}$ and the reward decomposes as

$$
r(x,a,\mu)=-\|\mu-\mu_\star\|_2^2-\lambda_{move}|a|,
\qquad
J(\theta)=\sum_{t=0}^{T} \mathbb{E}_{\mu_t,\pi_\theta}[r(X_t,A_t,\mu_t)].
$$

The diagnostic plots therefore focus on the state-time heatmap, target errors $\|\mu_t-\mu_\star\|_1$, $\|\mu_t-\mu_\star\|_2$, a ring $W_1$ proxy, and transport flux.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from mfc.experiments import notebook_helpers as nh

ENV_NAME = "distribution-planning"
BASE_DIR = ROOT / "runs" / "notebook_bundles" / ENV_NAME
PRESET = "smoke"
QUICK = PRESET == "smoke"
RUN_MISSING = True
FORCE_REBUILD = False
EXTENDED = True

In [ ]:
bundle = (
    nh.ensure_discrete_benchmark_bundle(ENV_NAME, BASE_DIR, quick=QUICK, force=FORCE_REBUILD, extended=EXTENDED, preset=PRESET)
    if RUN_MISSING
    else nh.bundle_paths(ENV_NAME, BASE_DIR)
)
bundle

## Figure Coverage

Goal: show which requested results from `docs/figures.md` are currently produced by this notebook and which artifacts support them.

The tables are an audit layer, not an estimator. A row maps a desired result family to the command or study that generates its data and to the notebook helper that renders it.

In [ ]:
nh.figure_checklist(ENV_NAME)

In [ ]:
nh.figure_coverage_matrix(ENV_NAME)

## Training: Simplex vs Logits

Goal: compare the optimization traces of the two finite-state perturbation geometries.

The value/objective plot estimates $J(\theta_k)$ at training episode $k$. The gradient plot tracks $\|\widehat g_k\|_2$, where $\widehat g_k$ is the MF-REINFORCE gradient estimate used by Adam.

Simplex uses affine law perturbations on the simplex; logits uses logistic-normal perturbations in logit coordinates. The curves should be compared using the same saved train/evaluation budget.

In [ ]:
histories = nh.load_training_histories(bundle)
nh.plot_training_comparison(histories)

## Application Diagnostics

Goal: inspect whether the policy moves mass toward the target distribution with controlled transport.

Reference: the second heatmap row and dashed metric curves use a model-based exact-flow policy optimized by differentiating the finite-state population recursion. This gives a strong numerical reference for target-tracking behavior.

The heatmap shows $\mu_t(x)$ over state and time. The error curves track

$$
\|\mu_t-\mu_\star\|_1,
\qquad
\|\mu_t-\mu_\star\|_2,
\qquad
W_1(\mu_t,\mu_\star)\ \text{(ring proxy)}.
$$

The flux panel summarizes how much mass is transported around the ring at each time.

In [ ]:
application = nh.load_application_data(bundle)
display(nh.reference_solution_table(ENV_NAME, application))
nh.plot_population_flow(application, ENV_NAME)
nh.plot_time_metrics(application, ENV_NAME)
nh.plot_policy_heatmaps(application, ENV_NAME)
nh.plot_discrete_application_details(application, ENV_NAME)

## Universal Diagnostics

Goal: validate the estimator chain before interpreting optimization performance.

The perturbation plots measure empirical geometry,

$$
d(M^\lambda,\mu),
$$

including quantile bands and local log-log slopes. The functional-law plots study

$$
\Gamma(M^\lambda)=(F_1(M^\lambda),\ldots,F_k(M^\lambda)),
\qquad
\frac{\Gamma(M^\lambda)-\Gamma(\mu)}{\lambda},
$$

which is the induced law of population signatures. The score plots check

$$
S_{t,\lambda}^\theta=\nabla_\theta\log q_{t,\lambda}^\theta(M_t),
\qquad \mathbb{E}[S_{t,\lambda}^\theta]\approx 0,
$$

and the gradient plots report bias, variance, MSE, norm ratio, and cosine agreement for an estimator $\widehat g$ against an oracle or reference gradient $g$:

$$
\operatorname{MSE}=\mathbb{E}\|\widehat g-g\|_2^2,
\qquad
\cos(\widehat g,g)=\frac{\widehat g\cdot g}{\|\widehat g\|_2\|g\|_2}.
$$

The sensitivity plots track errors in $D_t=\partial_\theta\Gamma(\mu_t^\theta)$, or in the finite-state case $D_t=\partial_\theta\mu_t^\theta$.

In [ ]:
diagnostics = nh.load_diagnostic_data(bundle)
nh.plot_perturbation_geometry(diagnostics)
nh.plot_perturbation_slopes(diagnostics)
nh.plot_functional_law(diagnostics)
nh.plot_functional_signature_means(diagnostics)
nh.plot_score_validation(diagnostics)
nh.plot_score_coordinate_diagnostics(diagnostics)
nh.plot_gradient_validation(diagnostics)
nh.plot_gradient_error_decomposition(diagnostics)
nh.plot_sensitivity_validation(diagnostics)
nh.plot_sensitivity_heatmap(diagnostics)

## Scaling, Budget, And Optimization Summaries

Goal: measure how estimator quality and optimization performance change with simulator budget, auxiliary budget, horizon, and training time.

The budget heatmap varies main and auxiliary samples $(B,n)$ under the approximate cost model

$$
C\approx C_{main}B+C_{aux}n.
$$

The horizon plot studies how gradient error changes with $T$, and the optimization plots compare objective or cost gaps against iteration count, runtime, and simulator-call budget.

In [ ]:
studies = nh.load_study_data(bundle)
grid_metrics = nh.load_study_grid_metrics(bundle)
optimization_history = nh.load_optimization_history(bundle)
nh.plot_budget_and_horizon(studies)
nh.plot_budget_pareto(studies, grid_metrics)
nh.plot_optimization_history(optimization_history)
nh.plot_optimization_summary(studies)

## Raw Tables For Custom Figures

Goal: expose the underlying CSV/JSON artifacts used by the plots so paper figures can be restyled or recomputed without rerunning training.

These tables are not new estimators. They are the saved values for population flows, policies, diagnostics, study grids, histories, and final metrics.

In [ ]:
application["simplex"]["time_metrics"].head(), diagnostics["simplex"]["gradient"].head(), studies["budget"].head()